In [1]:
import pandas as pd
from tqdm.notebook import tqdm
from pprint import pprint

In [2]:
file_export = 'data/base_wiki_pt_cleaned_2.pq'

In [149]:
df = pd.read_parquet(file_export, engine='fastparquet')

In [150]:
df.texto = df.texto.str.strip()
df.texto = df.texto.str.replace(r'\s+', ' ', regex=True)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_20314/161134553.py:2: SyntaxWarning: invalid escape sequence '\s'
  df.texto = df.texto.str.replace('\s+', ' ', regex=True)


In [151]:
df.head()

,index,texto
0,1031610,O Opirus é um automóvel sedan de porte grande ...
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...
2,649783,Eleutherodactylus cajamarcensis é uma espécie ...
3,1826843,Tricromatismo ou visão tricromática é a capaci...
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...


In [152]:
def formar_paragrafos(text, n=200, p=0.75):
    paragrafos = text.split('.')
    novo_paragrafo = ''
    for x in range(len(paragrafos)):
        proposto = (novo_paragrafo + f' {paragrafos[x]}').strip()
        if(x == 0 and len(proposto) < n):
            novo_paragrafo = proposto
            continue
        if len(proposto) >= int(n*p) and len(proposto) <= n:
            novo_paragrafo = proposto
            break
    return novo_paragrafo, len(novo_paragrafo)

In [153]:
tqdm.pandas(desc='Podando parágrafos')
n, p = 200, 0.75
df[['text_cut', 'len_text_cut']] = df.apply(lambda x: formar_paragrafos(x['texto'], n, p), result_type='expand', axis=1)

In [154]:
df.head()

,index,texto,text_cut,len_text_cut
0,1031610,O Opirus é um automóvel sedan de porte grande ...,O Opirus é um automóvel sedan de porte grande ...,173
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...,Costa Macedo Giraldes Barba de Noronha e Brito...,170
2,649783,Eleutherodactylus cajamarcensis é uma espécie ...,Eleutherodactylus cajamarcensis é uma espécie ...,82
3,1826843,Tricromatismo ou visão tricromática é a capaci...,Tricromatismo ou visão tricromática é a capaci...,151
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...,Ode do grego antigo ōidē é um poema de estilo ...,169


In [155]:
print('% de parágrafos extraídos:', (len(df[df.len_text_cut > (n*p)]) / len(df)))

% de parágrafos extraídos: 0.824197


In [156]:
n*p

150.0

In [157]:
df_filtered = df[df.len_text_cut > (n*p)]

In [158]:
df_filtered = df_filtered.drop(columns=['texto'])

In [159]:
df_filtered.to_parquet('data/base_wiki_pt_cleaned_cutted.pq', index=False)
df_filtered.head(10)

,index,text_cut,len_text_cut
0,1031610,O Opirus é um automóvel sedan de porte grande ...,173
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...,170
3,1826843,Tricromatismo ou visão tricromática é a capaci...,151
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...,169
7,1396393,Palinopsia grego: palin para novamente e opsia...,186
8,1911589,Sir Walter Lawry Buller 9 de Outubro de 1838 -...,176
10,1151976,Nome usado na lista do Património Mundial A pa...,180
11,1422950,Paulo em latim: Paulus; em grego:; romaniz O ...,156
12,1599893,Rua do Riachuelo hoje Rua Riachuelo é um logra...,185
13,663813,"A torção de talheres, ou ato de entortar colhe...",196


In [3]:
df_filtered = pd.read_parquet('data/base_wiki_pt_cleaned_cutted.pq')

In [6]:
indexes = df_filtered.sample(frac=1, random_state=42).index.tolist()
perc_train, perc_eval, perc_test = 0.8, .015, 0.215
qtde_train, qtde_eval, qtde_test = int(len(indexes)*perc_train), int(len(indexes)*perc_eval), int(len(indexes)*perc_test)
indexes_train = indexes[:qtde_train]
indexes_eval = indexes[qtde_train: qtde_train+qtde_eval]
indexes_test = indexes[qtde_train+qtde_eval: -1]

In [7]:
1-(0.8-.015)

0.21499999999999997

In [8]:
len(indexes_train), len(indexes_eval), len(indexes_test)

(659357, 12362, 152477)

In [9]:
df_train = df_filtered[df_filtered.index.isin(indexes_train)]
df_eval = df_filtered[df_filtered.index.isin(indexes_eval)]
df_test = df_filtered[df_filtered.index.isin(indexes_test)]

df_train.to_parquet('data/train_wiki_cleaned_cutted.pq', index=False)
df_eval.to_parquet('data/eval_wiki_cleaned_cutted.pq', index=False)
df_test.to_parquet('data/test_wiki_cleaned_cutted.pq', index=False)

In [10]:
all_texts = ''.join(df_train['text_cut'].values.tolist())

In [11]:
characters = set(all_texts)

In [12]:
len(characters)

475

In [13]:
characters = list(sorted(characters))

In [14]:
import tokenizer
import importlib
importlib.reload(tokenizer)

<module 'tokenizer' from '/home/alvarinho/Documentos/Estudos/RNN/tokenizer.py'>

In [15]:
tok = tokenizer.Tokenizer(characters, None)

In [16]:
text_test_enc = df_filtered.text_cut.values[-1]
pprint(text_test_enc)

('Ford Amphitheatre é uma arena multiuso localizada em Tampa, na Flórida, '
 'Estados Unidos  Em uma pesquisa on-line que acompanha um artigo sobre o '
 'assunto na St')


In [17]:
enc_text = tok.encode(text_test_enc)
pprint(enc_text)

[1,
 28,
 63,
 66,
 52,
 4,
 23,
 61,
 64,
 56,
 57,
 68,
 56,
 53,
 49,
 68,
 66,
 53,
 4,
 117,
 4,
 69,
 61,
 49,
 4,
 49,
 66,
 53,
 62,
 49,
 4,
 61,
 69,
 60,
 68,
 57,
 69,
 67,
 63,
 4,
 60,
 63,
 51,
 49,
 60,
 57,
 74,
 49,
 52,
 49,
 4,
 53,
 61,
 4,
 42,
 49,
 61,
 64,
 49,
 7,
 4,
 62,
 49,
 4,
 28,
 60,
 127,
 66,
 57,
 52,
 49,
 7,
 4,
 27,
 67,
 68,
 49,
 52,
 63,
 67,
 4,
 43,
 62,
 57,
 52,
 63,
 67,
 4,
 4,
 27,
 61,
 4,
 69,
 61,
 49,
 4,
 64,
 53,
 67,
 65,
 69,
 57,
 67,
 49,
 4,
 63,
 62,
 8,
 60,
 57,
 62,
 53,
 4,
 65,
 69,
 53,
 4,
 49,
 51,
 63,
 61,
 64,
 49,
 62,
 56,
 49,
 4,
 69,
 61,
 4,
 49,
 66,
 68,
 57,
 55,
 63,
 4,
 67,
 63,
 50,
 66,
 53,
 4,
 63,
 4,
 49,
 67,
 67,
 69,
 62,
 68,
 63,
 4,
 62,
 49,
 4,
 41,
 68,
 2]


In [18]:
dec_text = tok.decode(enc_text)
pprint(dec_text)

('<bos>Ford Amphitheatre é uma arena multiuso localizada em Tampa, na Flórida, '
 'Estados Unidos  Em uma pesquisa on-line que acompanha um artigo sobre o '
 'assunto na St<eos>')


In [19]:
import json

with open("artifacts/character_tokenizer.json", "w", encoding="utf-8") as f:
    json.dump(tok.to_dict(), f, ensure_ascii=False, indent=2)